# Embeddings **zembed-1** sur GPU (Google Colab)

Ce notebook encode les publications FSBM avec les **poids ouverts officiels de zembed-1**
(`zeroentropy/zembed-1-embedding`, Hugging Face, licence Apache-2.0, 4 Md de paramètres) sur le GPU gratuit de Colab.
Aucune clé API n'est nécessaire (ZeroEntropy n'accepte plus de nouvelles inscriptions). Le modèle est le **même** que
celui de l'API hébergée : seul le mode d'exécution change.

**Avant de commencer**
1. Sur ton PC : `python scripts/make_colab_bundle.py` → crée `outputs/colab/colab_bundle.zip`.
2. Colab : menu **Exécution → Modifier le type d'exécution → GPU (T4)**.
3. Exécute les cellules dans l'ordre ; envoie `colab_bundle.zip` quand la cellule 2 le demande.
4. À la fin, un fichier `zembed_outputs.zip` se télécharge : décompresse-le dans `data/vector_store/` de ton projet,
   puis lance `python scripts/05_build_index.py`.

In [ ]:
# 1) GPU disponible ?
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "⚠ Pas de GPU : Exécution → Modifier le type d'exécution")

In [ ]:
# 2) Envoi de l'archive du projet et décompression
from google.colab import files
import zipfile, os
uploaded = files.upload()                      # choisir colab_bundle.zip
name = next(iter(uploaded))
zipfile.ZipFile(name).extractall(".")
print("Fichiers :", sorted(os.listdir(".")))

In [ ]:
# 3) Dépendances (torch est déjà présent sur Colab)
!pip install -q "sentence-transformers>=5.0" pandas pyarrow PyYAML python-dotenv numpy

In [ ]:
# 4) Vérification : chargement des poids officiels (~8 Go, quelques minutes) et dimension attendue = 2560
!python scripts/04_generate_embeddings.py --transport local --device cuda --check

In [ ]:
# 5) Encodage de toutes les publications + des 5 requêtes de démonstration.
#    « auto » : bfloat16 sur GPU récent, float16 sur T4. Si des NaN apparaissent en float16, on retente en bfloat16.
import subprocess, sys
base = [sys.executable, "scripts/04_generate_embeddings.py", "--transport", "local", "--device", "cuda",
        "--with-queries", "--batch-size", "16", "--embed-in-json", "false"]
for extra in ([], ["--dtype", "bfloat16"]):
    result = subprocess.run(base + extra)
    if result.returncode == 0:
        break
    print("⚠ échec, nouvelle tentative avec", extra or "(rien)")
assert result.returncode == 0, "L'encodage a échoué : voir les messages ci-dessus"

In [ ]:
# 6) Résumé + archive à télécharger
import json, numpy as np, pandas as pd, zipfile
from google.colab import files
info = json.load(open("data/vector_store/embedding_run.json"))
print({k: info[k] for k in ("model", "transport", "runtime", "dimension", "n_embedded", "demo_queries_cached")})
vec = np.load("data/vector_store/embeddings.npy"); print("embeddings :", vec.shape, vec.dtype, "| NaN :", bool(np.isnan(vec).any()))
keep = ["embeddings.npy", "embedding_manifest.csv", "embedding_run.json", "embedding_cache.sqlite"]
with zipfile.ZipFile("zembed_outputs.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in keep:
        zf.write(f"data/vector_store/{f}", f)
files.download("zembed_outputs.zip")

## Ensuite, sur ton PC
```powershell
# décompresser zembed_outputs.zip dans data\vector_store\
python scripts\04_generate_embeddings.py --attach-only     # intègre les vecteurs dans dataset_final.json (si assez petit)
python scripts\05_build_index.py
python scripts\06_demo_search.py --researchers            # les 5 requêtes de démonstration sont déjà en cache : le modèle n'est pas rechargé
python scripts\07_semantic_map.py
python scripts\08_descriptive_analysis.py
```
Pour une **requête libre** (hors 5 requêtes de démo), le PC charge les poids en local (~8 Go de RAM, lent sur CPU) ;
sinon relance ce notebook avec `--query` ajouté, ou utilise Colab.